# Lesson 9.1: Study Design: How the Data Were Collected Decides What You Can Claim

Simulate 200 people with a hidden fitness level and see that a coin-flip assignment balances it between two groups while self-selection does not, which biases the estimated effect.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Create people with a hidden trait

We invent 200 people. Each has a fitness score that we, as researchers, would not normally see. The outcome (say, a health score) depends on fitness and on a program that truly adds 5 points. A seed makes the simulation repeatable.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
n = 200
true_effect = 5

fitness = rng.normal(50, 10, n)
noise = rng.normal(0, 5, n)
base_outcome = 40 + 0.5 * fitness + noise

print(f'people = {n}, mean fitness = {fitness.mean():.1f}')

people = 200, mean fitness = 49.7


## 2. Random assignment: a coin decides

Each person is put in the program or the comparison group by a coin flip, so fitness plays no part. With enough people, the two groups come out alike on fitness, and on every other trait we did not even measure.

In [2]:
random_group = rng.random(n) < 0.5

print('RANDOM ASSIGNMENT')
print(f'  program    : n = {random_group.sum()}, mean fitness = {fitness[random_group].mean():.1f}')
print(f'  comparison : n = {(~random_group).sum()}, mean fitness = {fitness[~random_group].mean():.1f}')

RANDOM ASSIGNMENT
  program    : n = 95, mean fitness = 49.0
  comparison : n = 105, mean fitness = 50.3


## 3. Self-selection: people choose

Now fitter people are more likely to sign up (a logistic curve turns fitness into a chance of choosing the program). The groups now differ on fitness before the program has done anything.

In [3]:
chance = 1 / (1 + np.exp(-(fitness - 50) / 5))
chose = rng.random(n) < chance

print('SELF-SELECTION')
print(f'  program    : n = {chose.sum()}, mean fitness = {fitness[chose].mean():.1f}')
print(f'  comparison : n = {(~chose).sum()}, mean fitness = {fitness[~chose].mean():.1f}')

SELF-SELECTION
  program    : n = 98, mean fitness = 54.6
  comparison : n = 102, mean fitness = 45.0


## 4. Compare the estimated effects

The program adds exactly 5 points to whoever gets it. Randomised groups should estimate about 5. In the self-selected groups, the program group was fitter to begin with, so the simple difference over-claims.

In [4]:
for label, group in [('random', random_group), ('self-selected', chose)]:
    outcome = base_outcome + true_effect * group
    diff = outcome[group].mean() - outcome[~group].mean()
    print(f'{label:14s} estimated effect = {diff:5.1f}   (true effect = {true_effect})')

random         estimated effect =   4.5   (true effect = 5)
self-selected  estimated effect =  10.7   (true effect = 5)


## Try this

- Change the sample size to 2000 and predict what happens to the gap in mean fitness between the random groups.
- Make the choice only weakly tied to fitness by dividing (fitness - 50) by 20 instead of 5. Predict how the self-selected estimate moves relative to the true effect of 5.

## The same idea in R, SPSS and Stata

**R**

```r
set.seed(42)
n <- 200
fitness <- rnorm(n, 50, 10)
base <- 40 + 0.5 * fitness + rnorm(n, 0, 5)
random <- runif(n) < 0.5
chose <- runif(n) < plogis((fitness - 50) / 5)
tapply(fitness, random, mean)
tapply(fitness, chose, mean)
mean((base + 5 * random)[random]) - mean((base + 5 * random)[!random])
mean((base + 5 * chose)[chose]) - mean((base + 5 * chose)[!chose])
```

**SPSS** (Transform > Compute Variable (with Random Number functions), then Analyze > Compare Means > Means)

```
SET SEED=42.
INPUT PROGRAM.
LOOP #i = 1 TO 200.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
COMPUTE fitness = RV.NORMAL(50,10).
COMPUTE random = (RV.UNIFORM(0,1) < 0.5).
COMPUTE chose = (RV.UNIFORM(0,1) < 1/(1+EXP(-(fitness-50)/5))).
MEANS TABLES=fitness BY random /CELLS=MEAN COUNT.
MEANS TABLES=fitness BY chose /CELLS=MEAN COUNT.
```

**Stata**

```
clear
set seed 42
set obs 200
generate fitness = rnormal(50, 10)
generate base = 40 + 0.5 * fitness + rnormal(0, 5)
generate random = runiform() < 0.5
generate chose = runiform() < invlogit((fitness - 50) / 5)
tabstat fitness, by(random) statistics(mean n)
tabstat fitness, by(chose) statistics(mean n)
generate outcome_r = base + 5 * random
generate outcome_c = base + 5 * chose
ttest outcome_r, by(random)
ttest outcome_c, by(chose)
```